<a href="https://colab.research.google.com/github/freyzamarshall02/w-okadavoicechangercolab/blob/main/Okada_Voice_Changer_Colab_Unofficial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="text-align: center;">
  <span style="color: #00ffff;">OKADA VOICE CHANGER COLAB (UNOFFICIAL BETA TEST)</span>
</h1>

<hr />
  <h2 style="text-align: center;">

  <h2 style="text-align: center;">
    <span style="color: #FFFFFF;">AI Realtime Voice Changer On Google Colab</span>

   <h2 style="text-align: center;">
    <span style="color: #FFFFFF;">Notebook By FreyzaMarshall</span>

  </h2>
  </span>
  </h2>
  <a href="https://trakteer.id/freyzamarshall" target="_parent"><img src="https://img.shields.io/badge/Trakteer-F16061?style=for-the-badge&logo=trakteer&logoColor=white" align="right" alt="Open"></a>

   <a href="https://discord.gg/sr5kyhRy3x" target="_parent"><img src="https://img.shields.io/badge/Discord-5865F2?style=for-the-badge&logo=discord&logoColor=white" align="right" alt="Open"></a>

  <a href="https://www.youtube.com/@FreyzaMarshall_" target="_parent"><img src="https://img.shields.io/badge/YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" align="right" alt="Open"></a>


For mobile user may experience bugs and feature limitations.
Join my discord if you have trouble.

In [ ]:
# @title **[1]** Check Google Colab VM Location

#@markdown Check the geographical location of your Google Colab virtual machine (VM).

#@markdown A VM located far from you may result in higher latency and slower network responses.

#@markdown **Note:** Google Colab typically assigns VM locations automatically, and you may not be able to choose the exact region.

import requests

try:
    response = requests.get("https://ipinfo.io/json", timeout=10)
    response.raise_for_status()
    data = response.json()

    ip = data.get("ip", "N/A")
    city = data.get("city", "N/A")
    region = data.get("region", "N/A")
    country = data.get("country", "N/A")

    print(f"Google Colab IP Address: {ip}")
    print(f"Town/City: {city}")
    print(f"Region: {region}")
    print(f"Country: {country}")

except requests.exceptions.RequestException as e:
    print(f"Could not retrieve location info: {e}")

In [ ]:
# @title **[2]** Clone repository and install dependencies
# @markdown This first step will download the latest version of Voice Changer and install the dependencies. **It can take some time to complete.**

import os
import subprocess
import time
import subprocess
import threading
import shutil
from IPython.display import clear_output

#@markdown ---
# @title **[Optional]** Connect to Google Drive
# @markdown Using Google Drive will automatically save your uploaded models for later use.

Use_Drive=False #@param {type:"boolean"}

# Install libportaudio2

print("\n" + "=" * 60)
print("🔄 Installing libportaudio2...")
print("=" * 60)
subprocess.run(['apt-get', '-y', 'install', '-qq', 'libportaudio2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['sudo', 'apt-get', '-qq', 'update'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['sudo', 'apt-get', 'install', '-qq', 'portaudio19-dev', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output()


# Install Webstuff

print("\n" + "=" * 60)
print("🔄 Installing Webstuff...")
print("=" * 60)
subprocess.run(['pip', 'install', 'git+https://github.com/freyzamarshall02/pyngrok.git'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
!npm install -g @hrzn/cli > /dev/null 2>&1
clear_output()

# Cloning repository

print("\n" + "=" * 60)
print("🔄 Cloning repository...")
print("=" * 60)
!git clone https://github.com/freyzamarshall02/voice-changer
%cd /content/voice-changer/server
clear_output()

# Installing Requirements

print("\n" + "=" * 60)
print("🔄 Installing requirements...")
print("=" * 60)
!pip install -r requirements.txt -q
clear_output()


# Use Drive?

if Use_Drive==True:
  if not os.path.exists('/content/drive'):
    print("\n" + "=" * 60)
    print("🔄 Mounting Drive...")
    print("=" * 60)
    from google.colab import drive
    drive.mount('/content/drive')

    !mkdir -p /content/drive/MyDrive/voice-changer/server/model_dir
    !rm -rf /content/voice-changer/server/model_dir

    drive_dir = "/content/drive/MyDrive/voice-changer/server/model_dir"
    colab_dir = "/content/voice-changer/server/model_dir"
    time.sleep(5)

    os.symlink(drive_dir,colab_dir,True)

clear_output()
print("\n" + "=" * 60)
print("✅ Dependencies Installed!")
print("=" * 60)


In [ ]:
# @title **Cell[3]** Start Server using **NGROK** or **HRZN**
# @markdown This cell will start the server, the first time that you run it will download the models, so it can take a while (~1-2 minutes)

import json
import sys
import os
import subprocess, threading, time, socket, urllib.request, portpicker, json
from IPython.display import clear_output, Javascript
from IPython.display import Audio, display

%cd /content/voice-changer/server

#======================Tunnels===========================

TUNNEL = "NGROK" #@param ["NGROK","HRZN"]

# @markdown ---
# @markdown You'll need a NGROK or HRZN account, but <font color=green>**it's free**</font> and easy to create!
# @markdown ---
# @markdown **1** - Create a <font color=green>**free**</font> account at [ngrok](https://dashboard.ngrok.com/signup) / [hrzn](https://hrzn.run/login) or **login with Google/Github account**\
# @markdown **2** - If you didn't logged in with Google/Github, you will need to **verify your e-mail**!\
# @markdown **3** - Get your [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) or [hrzn](https://hrzn.run/dashboard) auth token and place it here:
Token = '' # @param {type:"string"}
# @markdown **4** - *(OPTIONAL FOR NGROK)* Change to a region near to you\
# @markdown `Default Region: ap - Asia/Pacific (Singapore)`
Region = "ap - Asia/Pacific (Singapore)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)","eu - Europe (Frankfurt)", "in - India (Mumbai)","jp - Japan (Tokyo)","sa - South America (Sao Paulo)", "us - United States (Ohio)"]

#@markdown **5** - *(optional)* Other options:
ClearConsole = True  # @param {type:"boolean"}
PORT = portpicker.pick_unused_port()

#======================Setup Tunnel===========================

if TUNNEL == "NGROK":
    from pyngrok import conf, ngrok
    MyConfig = conf.PyngrokConfig()
    MyConfig.auth_token = Token
    MyConfig.region = Region[0:2]
    conf.set_default(MyConfig)
    ngrokConnection = ngrok.connect(PORT)
    public_url = ngrokConnection.public_url

elif TUNNEL == "HRZN":
    subprocess.run(['rm', '-rf', 'url.txt'])
    !hrzn login $Token
    os.system(f"hrzn tunnel http://localhost:{PORT} >> url.txt 2>&1 &")
    time.sleep(8)

    public_url = subprocess.getoutput(
        "grep -oE 'https://[a-zA-Z0-9.-]+\\.hrzn\\.run' url.txt"
    ).strip().strip('"').strip("'")

    if not public_url:
        raise RuntimeError("❌ Failed to get HRZN tunnel URL. Check that your token is correct.")

    print("--------- SERVER READY! ---------")
    print("Your server is available at:")
    print(public_url)
    print("---------------------------------")

#======================Start Server===========================

if TUNNEL == "NGROK":
    def wait_for_server():
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', PORT))
            if result == 0:
                break
            sock.close()
        if ClearConsole:
            clear_output()
        print("--------- SERVER READY! ---------")
        print("Your server is available at:")
        print(public_url)
        print("---------------------------------")

    threading.Thread(target=wait_for_server, daemon=True).start()

    !python3 Okada.py \
      -p {PORT} \
      --https False \
      --allowed-origins {public_url}

elif TUNNEL == "HRZN":
    !python3 Okada.py \
      -p {PORT} \
      --https False \
      --allowed-origins {public_url}

#======================Stop Server===========================

if TUNNEL == "NGROK":
    ngrok.disconnect(ngrokConnection.public_url)
    clear_output()
    print("--------- SERVER STOPPED! ---------")

elif TUNNEL == "HRZN":
    subprocess.run(['rm', '-rf', 'url.txt'])
    subprocess.run(['fuser', '-k', str(PORT)])
    clear_output()
    print("--------- SERVER STOPPED! ---------")
